<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/process/neqsim_compressor_capacity_bottleneck_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capacity-aware compressor optimization with NeqSim

This notebook demonstrates the NeqSim compressor-capacity integration introduced in
[NeqSim PR 2641](https://github.com/equinor/neqsim/pull/2641). A single set of physical
constraints is shared by:

- the compressor operating-point result;
- `ProcessSystem.findBottleneck()`; and
- `PressureBoundaryOptimizer`.

The example generates every compressor speed line together with the surge and stonewall
boundaries, maximizes throughput at fixed pressure boundaries, screens a driver upgrade, and
evaluates a field-life strategy in which inlet pressure is lowered as reservoir pressure declines.

The case is synthetic and intended for teaching and method development. Vendor maps, operating
limits, control logic, and uncertainty must be used for real equipment decisions.

## Learning objectives

After completing the notebook, you can:

1. build and solve a gas-compression process in NeqSim;
2. generate seven complete speed curves with head and efficiency data;
3. plot explicit surge and stonewall boundaries;
4. inspect typed, immutable operating-point and capacity snapshots;
5. add a dynamic vendor constraint to the standard NeqSim capacity model;
6. run maximum-throughput pressure-boundary optimization;
7. confirm that optimization and bottleneck reporting use the same constraint;
8. inspect every optimization iteration and constraint margin;
9. quantify the value of increasing driver capacity; and
10. combine facility capacity with declining-pressure well deliverability over field life.

## 1. One constraint model, three consumers

The integration removes a parallel compressor-constraint implementation from the pressure-boundary
optimizer. The optimizer now consumes enabled NeqSim `CapacityConstraint` objects directly.

| Consumer | Main question | Shared data |
|---|---|---|
| Operating-point result | Is this machine point physically feasible? | pressure target, map position, recycle, constraint snapshots |
| Bottleneck analysis | Which equipment constraint is closest to or beyond its limit? | utilization, margin, severity |
| Optimization | What is the highest feasible production rate? | the same enabled non-advisory constraints |

For a maximum constraint,

$$
U_{\max}=\frac{x}{x_{\max}},
$$

while a minimum-good margin such as surge or stonewall uses

$$
U_{\min}=\frac{x_{\min}}{x}.
$$

The common feasibility boundary is \(U \le 1\).

## 2. Reproducible pre-release setup

Until the API is included in a released NeqSim Python package, this notebook builds the exact
reviewed Java source commit. The custom JAR is placed ahead of the released dependency JAR before
the JVM starts. After release, replace this cell with a normal pinned `neqsim` installation.

The source build normally takes several minutes in a fresh Colab runtime.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import os
import shutil
import subprocess
import sys

PYTHON_NEQSIM_VERSION = "3.16.0"
NEQSIM_SOURCE_SHA = "ea393224e3a5be2a7cd198166017b0269d58ed1a"
SOURCE_ROOT = Path("/content/neqsim-capacity-source")

required = {
    "neqsim": PYTHON_NEQSIM_VERSION,
    "pandas": None,
    "matplotlib": None,
}
install_specs = []
for package_name, required_version in required.items():
    try:
        installed_version = version(package_name)
    except PackageNotFoundError:
        installed_version = None
    if installed_version is None:
        install_specs.append(
            package_name
            if required_version is None
            else f"{package_name}=={required_version}"
        )
    elif required_version and installed_version != required_version:
        install_specs.append(f"{package_name}=={required_version}")

if install_specs:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            *install_specs,
        ],
        check=True,
    )

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
subprocess.run(["git", "init", "--quiet"], cwd=SOURCE_ROOT, check=True)
subprocess.run(
    [
        "git",
        "remote",
        "add",
        "origin",
        "https://github.com/equinor/neqsim.git",
    ],
    cwd=SOURCE_ROOT,
    check=True,
)
subprocess.run(
    ["git", "fetch", "--quiet", "--depth", "1", "origin", NEQSIM_SOURCE_SHA],
    cwd=SOURCE_ROOT,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--quiet", "FETCH_HEAD"],
    cwd=SOURCE_ROOT,
    check=True,
)
subprocess.run(
    ["mvn", "-q", "-DskipTests", "package"],
    cwd=SOURCE_ROOT,
    check=True,
)

jar_candidates = [
    path
    for path in (SOURCE_ROOT / "target").glob("neqsim-*.jar")
    if not any(
        marker in path.name
        for marker in ("sources", "javadoc", "tests", "original")
    )
]
if not jar_candidates:
    raise FileNotFoundError("The NeqSim source build did not produce a runtime JAR.")
SOURCE_JAR = sorted(jar_candidates, key=lambda path: len(path.name))[0]
print("Built source:", NEQSIM_SOURCE_SHA)
print("Runtime JAR:", SOURCE_JAR)

In [ ]:
os.environ["NEQSIM_JVM_AUTOSTART"] = "0"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/neqsim-compressor-optimization")

import jpype

jpype.addClassPath(str(SOURCE_JAR))
from neqsim.neqsimpython import init_jvm

init_jvm()
from neqsim import jneqsim

import json
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

OperatingPointResult = jpype.JClass(
    "neqsim.process.equipment.compressor.CompressorOperatingPointResult"
)
loaded_from = str(
    OperatingPointResult.class_.getProtectionDomain()
    .getCodeSource()
    .getLocation()
)
assert NEQSIM_SOURCE_SHA
assert "neqsim-capacity-source" in loaded_from

VERSIONS = {
    "Python": sys.version.split()[0],
    "NeqSim Python package": version("neqsim"),
    "NeqSim Java source": NEQSIM_SOURCE_SHA,
    "Java class source": loaded_from,
}
display(pd.Series(VERSIONS, name="Value").to_frame())

plt.style.use("seaborn-v0_8-whitegrid")

## 3. Build a pressure-boundary compression process

The process contains a compositional gas feed, one export compressor, an aftercooler, and a delivery
stream. The compressor raises pressure from 50 to 100 bara. The design flow is used only to anchor
the generated map; the optimizer later changes the flow.

In [ ]:
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
CompressorChartGenerator = (
    jneqsim.process.equipment.compressor.CompressorChartGenerator
)
CapacityConstraint = (
    jneqsim.process.equipment.capacity.CapacityConstraint
)
ConstraintType = (
    jneqsim.process.equipment.capacity.CapacityConstraint.ConstraintType
)
ConstraintSeverity = (
    jneqsim.process.equipment.capacity.CapacityConstraint.ConstraintSeverity
)
PressureBoundaryOptimizer = (
    jneqsim.process.util.optimizer.PressureBoundaryOptimizer
)

DESIGN_FLOW_KG_H = 30_000.0
INLET_PRESSURE_BARA = 50.0
EXPORT_PRESSURE_BARA = 100.0

gas = SystemSrkEos(288.15, INLET_PRESSURE_BARA)
for component_name, mole_fraction in {
    "methane": 0.90,
    "ethane": 0.07,
    "propane": 0.03,
}.items():
    gas.addComponent(component_name, mole_fraction)
gas.setMixingRule("classic")
gas.setMultiPhaseCheck(False)

feed = Stream("Feed", gas)
feed.setFlowRate(DESIGN_FLOW_KG_H, "kg/hr")
feed.setPressure(INLET_PRESSURE_BARA, "bara")

compressor = Compressor("Export compressor", feed)
compressor.setOutletPressure(EXPORT_PRESSURE_BARA, "bara")
compressor.setUsePolytropicCalc(True)
compressor.setPolytropicEfficiency(0.76)

aftercooler = Cooler("Export aftercooler", compressor.getOutletStream())
aftercooler.setOutTemperature(303.15)
export_stream = Stream("Export", aftercooler.getOutletStream())

process = ProcessSystem()
for unit in (feed, compressor, aftercooler, export_stream):
    process.add(unit)
process.run()

baseline = pd.Series(
    {
        "Feed [kg/h]": feed.getFlowRate("kg/hr"),
        "Suction pressure [bara]": feed.getPressure("bara"),
        "Discharge pressure [bara]": compressor.getOutletStream().getPressure(
            "bara"
        ),
        "Power [kW]": compressor.getPower("kW"),
        "Head [kJ/kg]": compressor.getPolytropicHead("kJ/kg"),
        "Discharge temperature [°C]": (
            compressor.getOutletStream().getTemperature("C")
        ),
    },
    name="Design point",
)
display(baseline.to_frame())

## 4. Generate every curve and both map boundaries

`CompressorChartGenerator` creates seven speed lines with five points per line. The first point of
each speed line defines surge and the last defines stonewall. We retain all head and efficiency
curves rather than reducing the map to one design point.

In [ ]:
generator = CompressorChartGenerator(compressor)
generator.setChartType("interpolate and extrapolate")
generator.setImpellerDiameter(0.70)
generator.enableAdvancedCorrections(2)
chart = generator.generateCompressorChart("normal curves", 7)
compressor.setCompressorChart(chart)
compressor.setMinimumSpeed(float(chart.getMinSpeedCurve()))
compressor.setMaximumSpeed(float(chart.getMaxSpeedCurve()))
compressor.setLimitSpeed(True)
compressor.setSolveSpeed(True)
compressor.reinitializeCapacityConstraints()
process.run()

map_rows = []
speeds = [float(value) for value in chart.getSpeeds()]
flows = chart.getFlows()
heads = chart.getHeads()
efficiencies = chart.getPolytropicEfficiencies()
for curve_index, speed_rpm in enumerate(speeds):
    for point_index in range(len(flows[curve_index])):
        if point_index == 0:
            position = "surge"
        elif point_index == len(flows[curve_index]) - 1:
            position = "stonewall"
        else:
            position = "interior"
        map_rows.append(
            {
                "Speed [rpm]": speed_rpm,
                "Point": point_index,
                "Position": position,
                "Flow [m³/h]": float(flows[curve_index][point_index]),
                "Head [kJ/kg]": float(heads[curve_index][point_index]),
                "Efficiency [%]": float(
                    efficiencies[curve_index][point_index]
                ),
            }
        )

map_data = pd.DataFrame(map_rows)
display(
    map_data.groupby("Speed [rpm]")
    .agg(
        points=("Point", "count"),
        minimum_flow=("Flow [m³/h]", "min"),
        maximum_flow=("Flow [m³/h]", "max"),
        maximum_head=("Head [kJ/kg]", "max"),
        maximum_efficiency=("Efficiency [%]", "max"),
    )
    .round(3)
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14.0, 5.6))
for speed_rpm, speed_data in map_data.groupby("Speed [rpm]"):
    label = f"{speed_rpm:.0f} rpm"
    axes[0].plot(
        speed_data["Flow [m³/h]"],
        speed_data["Head [kJ/kg]"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        speed_data["Flow [m³/h]"],
        speed_data["Efficiency [%]"],
        marker="o",
        label=label,
    )

surge_data = map_data[map_data["Position"] == "surge"]
stonewall_data = map_data[map_data["Position"] == "stonewall"]
axes[0].plot(
    surge_data["Flow [m³/h]"],
    surge_data["Head [kJ/kg]"],
    "k--",
    linewidth=2.4,
    label="Surge boundary",
)
axes[0].plot(
    stonewall_data["Flow [m³/h]"],
    stonewall_data["Head [kJ/kg]"],
    color="firebrick",
    linestyle=":",
    linewidth=2.4,
    label="Stonewall boundary",
)
axes[0].scatter(
    [feed.getFlowRate("m3/hr")],
    [compressor.getPolytropicHead("kJ/kg")],
    s=110,
    color="gold",
    edgecolor="black",
    zorder=5,
    label="Design point",
)
axes[0].set(
    title="Head map with complete operating envelope",
    xlabel="Actual suction flow [m³/h]",
    ylabel="Polytropic head [kJ/kg]",
)
axes[1].set(
    title="Efficiency curves",
    xlabel="Actual suction flow [m³/h]",
    ylabel="Polytropic efficiency [%]",
)
for axis in axes:
    axis.legend(ncol=2, fontsize=8)
figure.tight_layout()
plt.show()

## 5. Inspect the shared capacity model

After a map is attached, the compressor provides speed, minimum speed, power, surge-margin, and
stonewall-margin constraints. Surge and stonewall are minimum-good constraints expressed in
physical percentage margin.

We also add a synthetic dynamic vendor driver limit. Its value supplier reads compressor power
after every process solve, so bottleneck detection and optimization see the current value rather
than a copied design-point number.

In [ ]:
baseline_power_kw = float(compressor.getPower("kW"))
VENDOR_POWER_LIMIT_KW = 1.18 * baseline_power_kw

power_supplier = jpype.JProxy(
    "java.util.function.DoubleSupplier",
    dict(getAsDouble=lambda: float(compressor.getPower("kW"))),
)
vendor_power = (
    CapacityConstraint("vendorDriverPower", "kW", ConstraintType.HARD)
    .setDesignValue(VENDOR_POWER_LIMIT_KW)
    .setMaxValue(VENDOR_POWER_LIMIT_KW)
    .setWarningThreshold(0.90)
    .setSeverity(ConstraintSeverity.HARD)
    .setDescription("Synthetic vendor driver shaft-power limit")
    .setValueSupplier(power_supplier)
)
compressor.addCapacityConstraint(vendor_power)


def capacity_table(machine):
    rows = []
    for entry in machine.getCapacityConstraints().entrySet():
        constraint = entry.getValue()
        rows.append(
            {
                "Constraint": str(constraint.getName()),
                "Type": str(constraint.getType()),
                "Severity": str(constraint.getSeverity()),
                "Current": float(constraint.getCurrentValue()),
                "Minimum": float(constraint.getMinValue()),
                "Design": float(constraint.getDesignValue()),
                "Maximum": float(constraint.getMaxValue()),
                "Utilization [%]": 100.0 * constraint.getUtilization(),
                "Margin [%]": 100.0 * constraint.getMargin(),
                "Enabled": bool(constraint.isEnabled()),
                "Violated": bool(constraint.isViolated()),
            }
        )
    return pd.DataFrame(rows)


design_constraints = capacity_table(compressor)
display(design_constraints.round(4))

## 6. Typed operating-point result

The result is a detached, serializable snapshot. It combines thermodynamic performance, actual and
requested pressure, map margins, anti-surge recycle screening, and copies of every capacity
constraint. It is suitable for time-series storage or transfer to an energy and emissions layer.

In [ ]:
design_result = compressor.getOperatingPointResult(0.02)

operating_point = pd.Series(
    {
        "Status": str(design_result.getOperatingStatus()),
        "Pressure status": str(design_result.getPressureTargetStatus()),
        "Feasible": bool(design_result.isFeasible()),
        "Flow [m³/h]": design_result.getFlowM3PerHour(),
        "Head [kJ/kg]": design_result.getPolytropicHeadKJPerKg(),
        "Speed [rpm]": design_result.getSpeedRpm(),
        "Power [kW]": design_result.getPowerKW(),
        "Surge margin [%]": 100.0 * design_result.getDistanceToSurge(),
        "Stonewall margin [%]": (
            100.0 * design_result.getDistanceToStonewall()
        ),
        "Required recycle [%]": (
            100.0 * design_result.getRequiredRecycleFraction()
        ),
        "Recycle power [kW]": design_result.getRecyclePowerLossKW(),
        "Limiting constraint": str(
            design_result.getLimitingConstraint()
        ),
        "Maximum utilization [%]": (
            100.0 * design_result.getMaximumCapacityUtilization()
        ),
    },
    name="Capacity-aware result",
)
display(operating_point.to_frame())

snapshot_rows = [
    {
        "Constraint": str(snapshot.getName()),
        "Current": snapshot.getCurrentValue(),
        "Unit": str(snapshot.getUnit()),
        "Utilization [%]": 100.0 * snapshot.getUtilization(),
        "Margin [%]": 100.0 * snapshot.getMargin(),
        "Violated": bool(snapshot.isViolated()),
    }
    for snapshot in design_result.getConstraints()
]
display(pd.DataFrame(snapshot_rows).round(4))

## 7. Maximum-throughput optimization

The optimizer varies feed rate while holding suction and delivery pressures. It now imports all
enabled non-design, non-advisory compressor constraints. The 15% minimum surge setting updates the
same physical `surgeMargin` constraint used by capacity and bottleneck reporting.

In [ ]:
optimizer = PressureBoundaryOptimizer(
    process,
    feed,
    export_stream,
)
optimizer.setAutoConfigureCompressors(False)
optimizer.setMinFlowRate(12_000.0)
optimizer.setMaxFlowRate(60_000.0)
optimizer.setRateUnit("kg/hr")
optimizer.setMinSurgeMargin(0.15)
optimizer.setPressureTolerance(0.01)
optimizer.setTolerance(0.001)
optimizer.setMaxIterations(45)

optimization = optimizer.findMaxFlowRate(
    INLET_PRESSURE_BARA,
    EXPORT_PRESSURE_BARA,
    "bara",
)

optimal_rate_kg_h = float(optimization.getOptimalRate())
feed.setPressure(INLET_PRESSURE_BARA, "bara")
feed.setFlowRate(optimal_rate_kg_h, "kg/hr")
compressor.setOutletPressure(EXPORT_PRESSURE_BARA, "bara")
process.run()

optimized_result = compressor.getOperatingPointResult(0.01)
process_bottleneck = process.findBottleneck()

optimization_summary = pd.Series(
    {
        "Feasible": bool(optimization.isFeasible()),
        "Maximum rate [kg/h]": optimal_rate_kg_h,
        "Compressor power [kW]": compressor.getPower("kW"),
        "Compressor speed [rpm]": compressor.getSpeed(),
        "Operating status": str(
            optimized_result.getOperatingStatus()
        ),
        "Typed limiting constraint": str(
            optimized_result.getLimitingConstraint()
        ),
        "Process bottleneck equipment": str(
            process_bottleneck.getEquipmentName()
        ),
        "Process bottleneck constraint": str(
            process_bottleneck.getConstraintName()
        ),
        "Bottleneck utilization [%]": (
            process_bottleneck.getUtilizationPercent()
        ),
        "Iterations": int(optimization.getIterations()),
    },
    name="Optimized point",
)
display(optimization_summary.to_frame())

In [ ]:
constraint_status_rows = [
    {
        "Constraint": str(status.getName()),
        "Severity": str(status.getSeverity()),
        "Margin": float(status.getMargin()),
        "Violated": bool(status.violated()),
        "Description": str(status.getDescription()),
    }
    for status in optimization.getConstraintStatuses()
]
constraint_status = pd.DataFrame(constraint_status_rows)
display(constraint_status.sort_values("Margin").round(5))

## 8. Audit the search trajectory

The optimizer returns every trial point. Plotting rate, feasibility, and bottleneck utilization
makes convergence and rejected points visible; it also provides a useful regression artifact.

In [ ]:
history_payload = json.loads(
    str(optimization.exportIterationHistoryAsJson())
)
iteration_history = pd.DataFrame(
    history_payload["iterationHistory"]
)
display(iteration_history)

figure, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
colors = np.where(iteration_history["feasible"], "#009E73", "#D55E00")
axes[0].scatter(
    iteration_history["iteration"],
    iteration_history["rate"],
    c=colors,
    s=60,
)
axes[0].axhline(
    optimal_rate_kg_h,
    color="black",
    linestyle="--",
    label="Reported optimum",
)
axes[0].set(
    title="Binary-feasibility search",
    xlabel="Iteration",
    ylabel="Trial feed rate [kg/h]",
)
axes[0].legend()

axes[1].plot(
    iteration_history["iteration"],
    100.0 * iteration_history["bottleneckUtilization"],
    marker="o",
)
axes[1].axhline(100.0, color="firebrick", linestyle="--")
axes[1].set(
    title="Bottleneck utilization",
    xlabel="Iteration",
    ylabel="Utilization [%]",
)
figure.tight_layout()
plt.show()

## 9. Debottlenecking: value of a larger driver

The dynamic vendor power constraint is changed without rebuilding the optimizer. Because the
optimizer retains the same `CapacityConstraint` object, each scenario uses the updated limit.
Incremental rate quantifies the production benefit of additional driver capacity, while the
reported bottleneck shows when a map or speed boundary replaces the driver as the limiting item.

In [ ]:
driver_limits_kw = baseline_power_kw * np.array([1.05, 1.18, 1.35, 1.60])
debottleneck_rows = []
for driver_limit_kw in driver_limits_kw:
    vendor_power.setDesignValue(float(driver_limit_kw))
    vendor_power.setMaxValue(float(driver_limit_kw))
    scenario = optimizer.findMaxFlowRate(
        INLET_PRESSURE_BARA,
        EXPORT_PRESSURE_BARA,
        "bara",
    )
    debottleneck_rows.append(
        {
            "Driver limit [kW]": driver_limit_kw,
            "Maximum rate [kg/h]": float(scenario.getOptimalRate()),
            "Feasible": bool(scenario.isFeasible()),
            "Bottleneck": (
                str(scenario.getBottleneck().getName())
                if scenario.getBottleneck() is not None
                else "None"
            ),
            "Bottleneck utilization [%]": (
                100.0 * scenario.getBottleneckUtilization()
            ),
        }
    )

vendor_power.setDesignValue(VENDOR_POWER_LIMIT_KW)
vendor_power.setMaxValue(VENDOR_POWER_LIMIT_KW)
debottleneck = pd.DataFrame(debottleneck_rows)
debottleneck["Incremental rate [kg/h]"] = (
    debottleneck["Maximum rate [kg/h]"]
    - debottleneck["Maximum rate [kg/h]"].iloc[0]
)
display(debottleneck.round(3))

figure, axis = plt.subplots(figsize=(8.0, 4.8))
axis.plot(
    debottleneck["Driver limit [kW]"],
    debottleneck["Maximum rate [kg/h]"],
    marker="o",
)
for _, row in debottleneck.iterrows():
    axis.annotate(
        row["Bottleneck"],
        (row["Driver limit [kW]"], row["Maximum rate [kg/h]"]),
        xytext=(5, 6),
        textcoords="offset points",
        fontsize=8,
    )
axis.set(
    title="Driver debottlenecking benefit and constraint switching",
    xlabel="Vendor driver limit [kW]",
    ylabel="Maximum feasible feed rate [kg/h]",
)
plt.show()

## 10. Field-life optimization with declining inlet pressure

A synthetic reservoir pressure decline is combined with a simple gas-well deliverability relation:

$$
\dot m_{\mathrm{well}}
=J\sqrt{p_{\mathrm{res}}^2-p_{\mathrm{in}}^2}.
$$

Reducing facility inlet pressure increases well potential, but it also increases compression head,
power, and map demand. Actual production is therefore:

$$
\dot m_{\mathrm{prod}}
=\min\left(
\dot m_{\mathrm{well}},
\dot m_{\mathrm{facility}}
\right).
$$

Two strategies are compared: a fixed 50 bara inlet and a scheduled reduction from 50 to 40 bara.
This is a screening model, not an inflow-performance or reservoir forecast.

In [ ]:
years = np.arange(2028, 2037)
reservoir_pressure_bara = np.linspace(95.0, 62.0, len(years))
inlet_strategies = {
    "Fixed 50 bara": np.full(len(years), 50.0),
    "Pressure reduction": np.linspace(50.0, 40.0, len(years)),
}
regularity = 0.94
productivity_index = DESIGN_FLOW_KG_H / math.sqrt(
    reservoir_pressure_bara[0] ** 2 - INLET_PRESSURE_BARA ** 2
)

field_rows = []
for strategy_name, inlet_schedule in inlet_strategies.items():
    cumulative_mass_mt = 0.0
    for year, reservoir_pressure, inlet_pressure in zip(
        years,
        reservoir_pressure_bara,
        inlet_schedule,
    ):
        pressure_square_difference = max(
            reservoir_pressure**2 - inlet_pressure**2,
            0.0,
        )
        well_potential_kg_h = (
            productivity_index * math.sqrt(pressure_square_difference)
        )
        facility_case = optimizer.findMaxFlowRate(
            float(inlet_pressure),
            EXPORT_PRESSURE_BARA,
            "bara",
        )
        facility_capacity_kg_h = (
            float(facility_case.getOptimalRate())
            if facility_case.isFeasible()
            else 0.0
        )
        production_kg_h = min(
            well_potential_kg_h,
            facility_capacity_kg_h,
        )
        cumulative_mass_mt += (
            production_kg_h * 8760.0 * regularity / 1.0e9
        )

        feed.setPressure(float(inlet_pressure), "bara")
        feed.setFlowRate(max(production_kg_h, 1.0), "kg/hr")
        compressor.setOutletPressure(EXPORT_PRESSURE_BARA, "bara")
        process.run()
        point = compressor.getOperatingPointResult(0.01)
        field_rows.append(
            {
                "Year": int(year),
                "Strategy": strategy_name,
                "Reservoir pressure [bara]": reservoir_pressure,
                "Inlet pressure [bara]": inlet_pressure,
                "Well potential [kg/h]": well_potential_kg_h,
                "Facility capacity [kg/h]": facility_capacity_kg_h,
                "Production [kg/h]": production_kg_h,
                "Power [kW]": point.getPowerKW(),
                "Surge margin [%]": 100.0 * point.getDistanceToSurge(),
                "Stonewall margin [%]": (
                    100.0 * point.getDistanceToStonewall()
                ),
                "Status": str(point.getOperatingStatus()),
                "Limiting constraint": str(
                    point.getLimitingConstraint()
                ),
                "Cumulative mass [Mt]": cumulative_mass_mt,
            }
        )

field_life = pd.DataFrame(field_rows)
display(field_life.round(3))

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(13.5, 9.0))
for strategy_name, strategy_data in field_life.groupby("Strategy"):
    axes[0, 0].plot(
        strategy_data["Year"],
        strategy_data["Production [kg/h]"],
        marker="o",
        label=strategy_name,
    )
    axes[0, 1].plot(
        strategy_data["Year"],
        strategy_data["Power [kW]"],
        marker="o",
        label=strategy_name,
    )
    axes[1, 0].plot(
        strategy_data["Year"],
        strategy_data["Inlet pressure [bara]"],
        marker="o",
        label=strategy_name,
    )
    axes[1, 1].plot(
        strategy_data["Year"],
        strategy_data["Cumulative mass [Mt]"],
        marker="o",
        label=strategy_name,
    )

axes[0, 0].set(title="Production", ylabel="Production [kg/h]")
axes[0, 1].set(title="Compression demand", ylabel="Power [kW]")
axes[1, 0].set(
    title="Facility inlet-pressure strategy",
    ylabel="Inlet pressure [bara]",
)
axes[1, 1].set(
    title="Cumulative produced mass",
    ylabel="Cumulative mass [Mt]",
)
for axis in axes.flat:
    axis.set_xlabel("Year")
    axis.legend()
figure.tight_layout()
plt.show()

In [ ]:
field_summary = (
    field_life.groupby("Strategy")
    .agg(
        cumulative_mass_mt=("Cumulative mass [Mt]", "max"),
        minimum_rate_kg_h=("Production [kg/h]", "min"),
        maximum_power_kw=("Power [kW]", "max"),
        minimum_surge_margin_percent=("Surge margin [%]", "min"),
        minimum_stonewall_margin_percent=("Stonewall margin [%]", "min"),
    )
)
reference_mass = field_summary.loc[
    "Fixed 50 bara",
    "cumulative_mass_mt",
]
field_summary["Incremental mass [Mt]"] = (
    field_summary["cumulative_mass_mt"] - reference_mass
)
field_summary["Incremental mass [%]"] = (
    100.0
    * field_summary["Incremental mass [Mt]"]
    / reference_mass
)
display(field_summary.round(4))

## 11. Engineering checks

These checks protect the example against silent changes in map generation, capacity semantics, and
optimizer wiring.

In [ ]:
surge_constraint = compressor.getCapacityConstraints().get("surgeMargin")
stonewall_constraint = compressor.getCapacityConstraints().get(
    "stonewallMargin"
)
snapshot_names = {
    str(snapshot.getName())
    for snapshot in design_result.getConstraints()
}
optimizer_constraint_names = set(constraint_status["Constraint"])

checks = {
    "seven speed lines generated": map_data["Speed [rpm]"].nunique() == 7,
    "five points on every speed line": (
        map_data.groupby("Speed [rpm]")["Point"].count() == 5
    ).all(),
    "surge is left of stonewall": (
        surge_data["Flow [m³/h]"].to_numpy()
        < stonewall_data["Flow [m³/h]"].to_numpy()
    ).all(),
    "surge uses minimum-good semantics": (
        surge_constraint.isMinimumConstraint()
    ),
    "stonewall uses minimum-good semantics": (
        stonewall_constraint.isMinimumConstraint()
    ),
    "typed result contains vendor constraint": (
        "vendorDriverPower" in snapshot_names
    ),
    "optimizer contains vendor constraint": any(
        name.endswith("_vendorDriverPower")
        for name in optimizer_constraint_names
    ),
    "optimizer contains surge constraint": any(
        name.endswith("_surgeMargin")
        for name in optimizer_constraint_names
    ),
    "optimizer contains stonewall constraint": any(
        name.endswith("_stonewallMargin")
        for name in optimizer_constraint_names
    ),
    "field-life schedules have all years": (
        field_life.groupby("Strategy")["Year"].count() == len(years)
    ).all(),
    "all production rates are non-negative": (
        field_life["Production [kg/h]"] >= 0.0
    ).all(),
}
check_table = pd.Series(checks, name="Passed").to_frame()
display(check_table)
assert all(checks.values())
print(f"All {len(checks)} engineering checks passed.")

## 12. Interpretation and next steps

The important result is architectural: compressor-map limits, vendor limits, bottleneck reporting,
and optimization now share one capacity model. This makes a maximum-rate result explainable: the
limiting constraint can be traced back to the same operating-point snapshot used for reporting or
for an eCalc time-series handoff.

The field-life section shows the central trade-off of inlet-pressure reduction. Lower pressure can
unlock well deliverability, but the value is capped by compressor head, speed, driver power,
surge/stonewall margin, and export-pressure feasibility.

For project work:

1. replace generated maps with complete vendor curves;
2. use measured driver power and temperature limits;
3. connect a reservoir or well model instead of the screening deliverability equation;
4. model parallel trains, availability, recycle control, and degradation;
5. add uncertainty bands around reservoir pressure, productivity, and equipment limits; and
6. send annual or hourly typed operating-point snapshots to eCalc for fuel and emissions accounting.

Related material:

- [NeqSim compressor documentation](https://equinor.github.io/neqsim/process/equipment/compressors.html)
- [NeqSim optimization and constraints](https://github.com/equinor/neqsim/blob/master/docs/process/optimization/OPTIMIZATION_AND_CONSTRAINTS.md)
- [Integrated NeqSim + eCalc field-life study](../power/neqsim_ecalc_integrated_energy_emissions.ipynb)
- [eCalc documentation](https://equinor.github.io/ecalc/docs/about/)